# Phase 5 — Feature Selection

Reduces the feature space from ~40 monthly aggregates to a stable, non-redundant subset.

## Pipeline
```
Stage 1 — Unsupervised (label-free, fast)
  VarianceFilter      Remove near-zero variance + quasi-constant
  CorrelationCluster  Remove redundant correlated features (|r| > 0.92)

Stage 2 — Supervised (3 methods, consensus vote)
  SHAPRanker          LightGBM SHAP mean|SHAP|
  BorutaSelector      Shadow feature comparison (20 trials)
  MRMRSelector        Min Redundancy Max Relevance
  → Keep if selected by >= 2/3 methods

Stage 3 — Optional Stability
  BootstrapStabilityAnalyzer  20 bootstrap runs
  VintageStabilityAnalyzer    Jan–Jul 2024 observation points
```

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_selection import FeatureSelectionPipeline
from src.feature_selection.stability import VintageStabilityAnalyzer
from src.utils import setup_logging
from data.sample.generate_sample import generate_sample_training_data

setup_logging('INFO')

## Load Training Data

In [ ]:
X_train, y_train = generate_sample_training_data(n=5000, seed=42)
print(f'Features: {X_train.shape}')
print(f'Segment distribution:\n{X_train.segment.value_counts()}')

## Run Feature Selection Pipeline

In [ ]:
pipeline = FeatureSelectionPipeline(
    config_path='../config/config.yaml',
    run_boruta=True,
    run_stability=False,   # Enable for full stability analysis
)
pipeline.fit(X_train, y_train, segment_col='segment')
print(pipeline.summary())

## Feature Selection Report

In [ ]:
report = pipeline.report()
print(f'Final selected features ({len(pipeline.final_features_)}):')
print(report[report.final_selected].to_string())

## SHAP Importance — Top Features

In [ ]:
shap_report = pipeline.shap_ranker_.report()
top20 = shap_report.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20['feature'][::-1], top20['mean_abs_shap'][::-1],
        color=['#2ecc71' if s else '#e74c3c' for s in top20['selected'][::-1]])
ax.set_xlabel('Mean |SHAP| value')
ax.set_title('Top 20 Features by SHAP Importance')
plt.tight_layout()
plt.show()

## Consensus Vote Heatmap

In [ ]:
vote_cols = [c for c in report.columns if c.startswith('selected_by_')]
top_features = report[report.vote_count >= 2].copy()

vote_matrix = top_features.set_index('feature')[vote_cols].astype(int)
vote_matrix.columns = [c.replace('selected_by_', '') for c in vote_cols]

fig, ax = plt.subplots(figsize=(8, max(4, len(vote_matrix) * 0.35)))
sns.heatmap(vote_matrix, annot=True, cbar=False, cmap='YlGn', ax=ax)
ax.set_title('Feature Selection Consensus (1 = selected by method)')
plt.tight_layout()
plt.show()

## Segment-Specific Feature Selection

In [ ]:
seg_features = pipeline.fit_per_segment(
    X_train, y_train, segment_col='segment', combine='union'
)
for seg, feats in seg_features.items():
    print(f'{seg}: {len(feats)} features — {feats[:5]}...')

## Correlation Cluster Report

In [ ]:
clusters = pipeline.corr_cluster_.report()
multi_clusters = clusters[clusters.representative != clusters.feature]
print(f'Redundant features removed by correlation: {len(multi_clusters)}')
print(multi_clusters.groupby('representative').feature.apply(list))